In [1]:
import pandas as pd
import sqlite3
import os
print(os.getcwd())

c:\Users\ayman\Documents\Github\fraud_detection\Ayman_folder\sql_financial_analysis


In [2]:
# data 
trans_df = pd.read_csv("../../data/trans.csv", sep=";", low_memory=False)
loan_df = pd.read_csv("../../data/loan.csv", sep=";")
account_df = pd.read_csv("../../data/account.csv", sep=";")
client_df = pd.read_csv("../../data/client.csv", sep=";")

In [3]:
# SQL database
conn = sqlite3.connect(':memory:')

# Creating variable for SQL quiery
trans_df.to_sql('transactions', conn, index=False)
loan_df.to_sql('loans', conn, index=False)
account_df.to_sql('accounts', conn, index=False)
client_df.to_sql('clients', conn, index=False)

5369

In [4]:
# Printing lenth, rows of each dataset to see what im working with 
print(f"Transactions: {len(trans_df):,} total rows")
print(f"Loans: {len(loan_df):,} total rows")
print(f"Accounts: {len(account_df):,} total rows")
print(f"Clients: {len(client_df):,} total rows")

Transactions: 1,056,320 total rows
Loans: 682 total rows
Accounts: 4,500 total rows
Clients: 5,369 total rows


In [5]:
# columns list
print(trans_df.columns.tolist())
print(trans_df.head(3))

['trans_id', 'account_id', 'date', 'type', 'operation', 'amount', 'balance', 'k_symbol', 'bank', 'account']
   trans_id  account_id    date    type operation  amount  balance k_symbol  \
0    695247        2378  930101  PRIJEM     VKLAD   700.0    700.0      NaN   
1    171812         576  930101  PRIJEM     VKLAD   900.0    900.0      NaN   
2    207264         704  930101  PRIJEM     VKLAD  1000.0   1000.0      NaN   

  bank  account  
0  NaN      NaN  
1  NaN      NaN  
2  NaN      NaN  


# Checking for NULL values, Duplicates and Outliers

In [19]:
# Checking for NULL values
null_check = pd.read_sql("""
    SELECT 
        COUNT(*) as total,
        COUNT(amount) as non_null_amount,
        COUNT(*) - COUNT(amount) as missing_amount,
        COUNT(type) as non_null_type,
        COUNT(*) - COUNT(type) as missing_type
    FROM transactions
""", conn)
print(null_check)

     total  non_null_amount  missing_amount  non_null_type  missing_type
0  1056320          1056320               0        1056320             0


In [20]:
# Checking for duplicates
dup_check = pd.read_sql("""
    SELECT account_id, date, amount, COUNT(*) as count
    FROM transactions
    GROUP BY account_id, date, amount
    HAVING COUNT(*) > 1
    ORDER BY count DESC
    LIMIT 10
""", conn)
print(dup_check)

   account_id    date  amount  count
0         541  980116  2400.0      3
1           9  960317   800.0      2
2          19  950430    47.1      2
3          19  950531   125.2      2
4          19  950930    98.0      2
5          54  970120   720.0      2
6          73  960110  3200.0      2
7         100  950105  1920.0      2
8         103  970228   195.9      2
9         103  970630   170.7      2


In [30]:
# Checking for outliers
outliers = pd.read_sql("""
    SELECT 
        MIN(amount) as min_amount,
        MAX(amount) as max_amount,
        ROUND(AVG(amount), 2) as avg_amount
    FROM transactions
""", conn)

print(outliers)

# zero amount is interesting to investigate

   min_amount  max_amount  avg_amount
0         0.0     87400.0     5924.15


In [31]:
# How many zero amount transactions?
zeros = pd.read_sql("""
    SELECT COUNT(*) as zero_transactions
    FROM transactions
    WHERE amount = 0
""", conn)

print(zeros)

   zero_transactions
0                 14


## Data Quality Summary

- Total records: 1,056,320 transactions
- NULL values: None found in key columns
- Duplicate transactions: None found
- Zero amount transactions: 14 (0.001%) — negligible, likely system entries
- Max transaction: 87,400 CZK — within reasonable range
- **Conclusion: Dataset is clean and ready for analysis**

 How many transactions are in the dataset total?

In [6]:
q1 = pd.read_sql("""
    SELECT COUNT(*) as total_transactions
    FROM transactions
""",conn)
print(q1)

   total_transactions
0             1056320


What is the average transaction amount ?

In [7]:
q2 = pd.read_sql("""
    SELECT AVG(amount) as average_transaction
    FROM transactions
""",conn)
print(q2)

   average_transaction
0          5924.145676


How many unique accounts are there?

In [8]:
q3 = pd.read_sql("""
    SELECT COUNT( DISTINCT account_id) as unique_accounts
    FROM transactions
""",conn)
print(q3)

   unique_accounts
0             4500


In [9]:
# columns list
print(trans_df.columns.tolist())
print(trans_df.head(3))

['trans_id', 'account_id', 'date', 'type', 'operation', 'amount', 'balance', 'k_symbol', 'bank', 'account']
   trans_id  account_id    date    type operation  amount  balance k_symbol  \
0    695247        2378  930101  PRIJEM     VKLAD   700.0    700.0      NaN   
1    171812         576  930101  PRIJEM     VKLAD   900.0    900.0      NaN   
2    207264         704  930101  PRIJEM     VKLAD  1000.0   1000.0      NaN   

  bank  account  
0  NaN      NaN  
1  NaN      NaN  
2  NaN      NaN  


Total amount of money transacted across all accounts?

In [10]:
q4 = pd.read_sql("""
    SELECT ROUND(SUM(amount), 2) as total_amount
    FROM transactions
""", conn)
print(q4)

   total_amount
0  6.257794e+09


How many transactions are of each type?

In [11]:
q5 = pd.read_sql("""
    SELECT type, COUNT(*) as total_transaction
    FROM transactions
    GROUP BY type
""", conn)
print(q5)

     type  total_transaction
0  PRIJEM             405083
1   VYBER              16666
2   VYDAJ             634571


In [12]:
# query 1 top 10 accounts by total transaction volume
q6 = pd.read_sql_query("""
    SELECT 
    account_id, 
    COUNT(*) AS num_transactions,
    ROUND(SUM(amount), 2) AS total_amount,
    ROUND(AVG(amount), 2) AS avg_transaction,
    ROUND(MAX(amount), 2) AS max__transaction
FROM transactions
GROUP BY account_id
ORDER BY total_amount DESC
LIMIT 10
""", conn)

In [13]:
print(q6)

   account_id  num_transactions  total_amount  avg_transaction  \
0         212               432     7619102.4         17636.81   
1        3521               429     7401229.2         17252.28   
2        2762               634     7399357.6         11670.91   
3        1132               414     7386440.3         17841.64   
4        2838               567     7365804.7         12990.84   
5         456               387     7338613.4         18962.83   
6        2219               488     7333024.6         15026.69   
7        1032               401     7327494.1         18273.05   
8        5228               454     7117524.8         15677.37   
9        9833               547     7037938.1         12866.43   

   max__transaction  
0           74385.0  
1           64300.0  
2           64600.0  
3           72147.0  
4           63400.0  
5           71746.0  
6           63900.0  
7           74648.0  
8           74522.0  
9           73224.0  


# KPI

A KPI is a specific measurable value that shows how well a business or team is achieving its goals. For example, if a company wants to grow revenue, their KPI might be monthly recurring revenue. If they want to retain customers, their KPI might be churn rate. As a data analyst my job is to track those KPIs, build dashboards around them, and flag when something looks off so leadership can act on it.

In [32]:
# KPI - Loan Default Rate
kpi = pd.read_sql("""
    SELECT 
        COUNT(*) as total_loans,
        SUM(CASE WHEN status IN ('B', 'D') THEN 1 ELSE 0 END) as defaulted_loans,
        ROUND(100.0 * SUM(CASE WHEN status IN ('B', 'D') THEN 1 ELSE 0 END) / COUNT(*), 2) as default_rate_pct
    FROM loans
""", conn)
print(kpi)

   total_loans  defaulted_loans  default_rate_pct
0          682               76             11.14


How many loans does each district have, and what is each district's default rate?

In [15]:
q6 = pd.read_sql("""
    SELECT a.district_id, l.status
    FROM loans l
    JOIN accounts a ON l.account_id = a.account_id
""", conn)

print(q6)

     district_id status
0             30      B
1             46      A
2             45      A
3             12      A
4              1      A
..           ...    ...
677           21      C
678           55      C
679            3      C
680           70      C
681           60      C

[682 rows x 2 columns]


In [33]:
q6 = pd.read_sql("""
    SELECT 
        a.district_id,
        COUNT(*) as total_loans,
        SUM(CASE WHEN l.status IN ('B', 'D') THEN 1 ELSE 0 END) as defaulted_loans,
        ROUND(100.0 * SUM(CASE WHEN l.status IN ('B', 'D') THEN 1 ELSE 0 END) / COUNT(*), 2) as default_rate_pct
    FROM loans l
    JOIN accounts a ON l.account_id = a.account_id
    GROUP BY a.district_id
    ORDER BY default_rate_pct DESC
    LIMIT 10
""", conn)

print(q6)

# Districts 67, 30, 22, and 20 have a 50% default rate —  significantly above the 11.14% overall average. 
# These districts should be flagged for stricter loan approval criteria.

   district_id  total_loans  defaulted_loans  default_rate_pct
0           67            6                3             50.00
1           30            2                1             50.00
2           22            2                1             50.00
3           20            6                3             50.00
4           73            8                3             37.50
5           25            3                1             33.33
6            6            9                3             33.33
7            3            6                2             33.33
8           21            7                2             28.57
9           69            8                2             25.00


Find all accounts whose total spending is above the average spending of all accounts?

In [34]:
# Subquery - Accounts spending above average
subquery = pd.read_sql("""
    SELECT account_id, 
           ROUND(SUM(amount), 2) as total_spent
    FROM transactions
    GROUP BY account_id
    HAVING SUM(amount) > (
        SELECT AVG(total) 
        FROM (
            SELECT SUM(amount) as total 
            FROM transactions 
            GROUP BY account_id
        )
    )
    ORDER BY total_spent DESC
    LIMIT 10
""", conn)
print(subquery)

   account_id  total_spent
0         212    7619102.4
1        3521    7401229.2
2        2762    7399357.6
3        1132    7386440.3
4        2838    7365804.7
5         456    7338613.4
6        2219    7333024.6
7        1032    7327494.1
8        5228    7117524.8
9        9833    7037938.1


## Subquery — High Value Accounts

Accounts spending above the portfolio average are prime candidates 
for premium banking products and relationship management. 
Account 212 leads with 7.6M CZK in total transactions.
These accounts should be monitored closely for both 
retention and fraud risk.

categorize accounts into spending tiers — Low, Medium, High

In [38]:
# Account Spending Segmentation using CASE WHEN
segments = pd.read_sql("""
    SELECT 
        account_id,
        ROUND(SUM(amount), 2) as total_spent,
        CASE 
            WHEN SUM(amount) < 1000000 THEN 'Low Spender'
            WHEN SUM(amount) BETWEEN 1000000 AND 4000000 THEN 'Medium Spender'
            WHEN SUM(amount) > 4000000 THEN 'High Spender'
        END as spending_segment
    FROM transactions
    GROUP BY account_id
    ORDER BY total_spent DESC
    LIMIT 20
""", conn)
print(segments)

# Count accounts per segment
segment_counts = pd.read_sql("""
    SELECT 
        CASE 
            WHEN SUM(amount) < 1000000 THEN 'Low Spender'
            WHEN SUM(amount) BETWEEN 1000000 AND 4000000 THEN 'Medium Spender'
            WHEN SUM(amount) > 4000000 THEN 'High Spender'
        END as spending_segment,
        COUNT(*) as total_accounts
    FROM transactions
    GROUP BY account_id
    ORDER BY total_accounts DESC
""", conn)

# Group by segment
print(segment_counts.groupby('spending_segment')['total_accounts'].count())

    account_id  total_spent spending_segment
0          212    7619102.4     High Spender
1         3521    7401229.2     High Spender
2         2762    7399357.6     High Spender
3         1132    7386440.3     High Spender
4         2838    7365804.7     High Spender
5          456    7338613.4     High Spender
6         2219    7333024.6     High Spender
7         1032    7327494.1     High Spender
8         5228    7117524.8     High Spender
9         9833    7037938.1     High Spender
10        3674    6941710.4     High Spender
11        5129    6939091.1     High Spender
12        5270    6937453.1     High Spender
13        3675    6936835.1     High Spender
14        2486    6910402.4     High Spender
15        4318    6872363.4     High Spender
16         862    6774346.9     High Spender
17        2915    6769370.2     High Spender
18          96    6749839.0     High Spender
19        1286    6610663.1     High Spender
spending_segment
High Spender       253
Low Spender    

## Customer Segmentation

- High Spenders (5.6%) drive the most transaction volume 
  and should be prioritized for premium services
- Medium Spenders (40.8%) represent growth opportunity — 
  targeted campaigns could move them into High tier
- Low Spenders (53.5%) are the majority but contribute 
  least to revenue — retention focus needed

Recommendation: Focus relationship management resources 
on the 253 High Spender accounts while developing 
upgrade campaigns for Medium Spenders.